Importaciones

In [1]:
!pip install -q transformers datasets evaluate sentencepiece sacrebleu accelerate

In [2]:
import time
import torch
import pandas as pd

from datasets import load_dataset
from transformers import pipeline
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

In [3]:
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    print("GPU no detectada")

GPU detectada: Tesla T4


Carga de dataset

In [4]:
dataset = load_dataset("Helsinki-NLP/opus_books", "en-es")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 93470
    })
})


In [6]:
#observacion del dataset
df = dataset["train"].select(range(4, 20)).to_pandas()

df["en"] = df["translation"].apply(lambda x: x["en"])
df["es"] = df["translation"].apply(lambda x: x["es"])

df[["id", "en", "es"]]

,id,en,es
0,4,The family of Dashwood had long been settled i...,La familia Dashwood llevaba largo tiempo afinc...
1,5,"Their estate was large, and their residence wa...","Su propiedad era de buen tamaño, y en el centr..."
2,6,The late owner of this estate was a single man...,El último dueño de esta propiedad había sido u...
3,7,"But her death, which happened ten years before...","Pero la muerte de ella, ocurrida diez años ant..."
4,8,"In the society of his nephew and niece, and th...","En compañía de su sobrino y sobrina, y de los ..."
5,9,His attachment to them all increased.,Su apego a todos ellos fue creciendo con el “t...
6,10,The constant attention of Mr. and Mrs. Henry D...,La constante atención que el señor Henry Dashw...
7,11,"By a former marriage, Mr. Henry Dashwood had o...","De un matrimonio anterior, el señor Henry Dash..."
8,12,"The son, a steady respectable young man, was a...","El hijo, un joven serio y respetable, tenía el..."
9,13,"By his own marriage, likewise, which happened ...","Además, su propio matrimonio, ocurrido poco de..."


In [7]:
#muestras
sample_size = 100
sample_dataset = dataset["train"].select(range(sample_size))

In [8]:
#Extraer los textos y refrencias
source_texts = [item["translation"]["en"] for item in sample_dataset]
reference_texts = [item["translation"]["es"] for item in sample_dataset]

print("Cantidad de textos fuente:", len(source_texts))
print("Cantidad de referencias:", len(reference_texts))

print("\nTexto original en inglés:")
print(source_texts[0])

print("\nReferencia en español:")
print(reference_texts[0])

Cantidad de textos fuente: 100
Cantidad de referencias: 100

Texto original en inglés:
Source: Project GutenbergAudiobook available here

Referencia en español:
Source: Wikisource & librodot.com


Carga del modelo m2m100_418

In [9]:
model_name = "facebook/m2m100_418M"

tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

device = device
model.to(device)

print("Modelo cargado:", model_name)
print("Dispositivo:", device)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modelo cargado: facebook/m2m100_418M
Dispositivo: 0


Pruebas

In [10]:
text = source_texts[6]

tokenizer.src_lang = "en"
encoded = tokenizer(text, return_tensors="pt", truncation=True).to(device)

generated_tokens = model.generate(
    **encoded,
    forced_bos_token_id=tokenizer.get_lang_id("es")
)

translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

print("Texto original:")
print(text)
print("\nTraducción generada:")
print(translated_text)

Texto original:
The late owner of this estate was a single man, who lived to a very advanced age, and who for many years of his life, had a constant companion and housekeeper in his sister.

Traducción generada:
El último propietario de esta propiedad era un hombre solitario, que vivió hasta una edad muy avanzada, y que durante muchos años de su vida, tenía un constante compañero y cuidador en su hermana.


In [11]:
#traduccion de la muestra
predictions = []

inicio = time.time()

for text in source_texts:
    tokenizer.src_lang = "en"
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id("es")
    )

    translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    predictions.append(translated_text)

fin = time.time()
tiempo_total = fin - inicio

print("Cantidad de traducciones generadas:", len(predictions))
print("Tiempo total de inferencia:", tiempo_total, "segundos")
print("Tiempo promedio por traducción:", tiempo_total / len(predictions), "segundos")

Cantidad de traducciones generadas: 100
Tiempo total de inferencia: 97.51730442047119 segundos
Tiempo promedio por traducción: 0.975173044204712 segundos


In [12]:
#Comparaciones
for i in range(5):
    print(f"Ejemplo {i+1}")
    print("EN:", source_texts[i])
    print("REF:", reference_texts[i])
    print("PRED:", predictions[i])
    print("-" * 80)

Ejemplo 1
EN: Source: Project GutenbergAudiobook available here
REF: Source: Wikisource & librodot.com
PRED: Fuente: Proyecto GutenbergAudiobook disponible aquí
--------------------------------------------------------------------------------
Ejemplo 2
EN: Sense and Sensibility
REF: SENTIDO Y SENSIBILIDAD
PRED: Sensación y sensibilidad
--------------------------------------------------------------------------------
Ejemplo 3
EN: Jane Austen
REF: JANE AUSTEN
PRED: de Jane Austen
--------------------------------------------------------------------------------
Ejemplo 4
EN: CHAPTER 1
REF: CAPITULO I
PRED: CAPÍTULO 1
--------------------------------------------------------------------------------
Ejemplo 5
EN: The family of Dashwood had long been settled in Sussex.
REF: La familia Dashwood llevaba largo tiempo afincada en Sussex.
PRED: La familia de Dashwood se había establecido en Sussex durante mucho tiempo.
--------------------------------------------------------------------------------


BLEU [BiLingual Evaluation
Understudy]

In [13]:
bleu = evaluate.load("sacrebleu")

references_for_bleu = [[ref] for ref in reference_texts]

bleu_result = bleu.compute(
    predictions=predictions,
    references=references_for_bleu
)

print("BLEU:", bleu_result["score"])
print(bleu_result)

BLEU: 25.37794406331642
{'score': 25.37794406331642, 'counts': [1680, 867, 508, 322], 'totals': [2900, 2800, 2700, 2602], 'precisions': [57.93103448275862, 30.964285714285715, 18.814814814814813, 12.375096079938508], 'bp': 0.998277347540926, 'sys_len': 2900, 'ref_len': 2905}


Tabla de comparaciones

In [14]:
results_df = pd.DataFrame({
    "source_en": source_texts,
    "reference_es": reference_texts,
    "prediction_es": predictions
})

results_df.head(10)

,source_en,reference_es,prediction_es
0,Source: Project GutenbergAudiobook available here,Source: Wikisource & librodot.com,Fuente: Proyecto GutenbergAudiobook disponible...
1,Sense and Sensibility,SENTIDO Y SENSIBILIDAD,Sensación y sensibilidad
2,Jane Austen,JANE AUSTEN,de Jane Austen
3,CHAPTER 1,CAPITULO I,CAPÍTULO 1
4,The family of Dashwood had long been settled i...,La familia Dashwood llevaba largo tiempo afinc...,La familia de Dashwood se había establecido en...
5,"Their estate was large, and their residence wa...","Su propiedad era de buen tamaño, y en el centr...","Su propiedad era grande, y su residencia se en..."
6,The late owner of this estate was a single man...,El último dueño de esta propiedad había sido u...,El último propietario de esta propiedad era un...
7,"But her death, which happened ten years before...","Pero la muerte de ella, ocurrida diez años ant...","Pero su muerte, que sucedió diez años antes de..."
8,"In the society of his nephew and niece, and th...","En compañía de su sobrino y sobrina, y de los ...","En la sociedad de su sobrino y nieta, y sus hi..."
9,His attachment to them all increased.,Su apego a todos ellos fue creciendo con el “t...,Su afecto a todos ellos aumentó.


Guardar metricas

In [15]:
summary_df = pd.DataFrame([{
    "modelo": model_name,
    "tamano": "418M",
    "num_ejemplos": len(source_texts),
    "bleu": bleu_result["score"],
    "tiempo_total_seg": tiempo_total,
    "tiempo_promedio_seg": tiempo_total / len(source_texts)
}])

summary_df

,modelo,tamano,num_ejemplos,bleu,tiempo_total_seg,tiempo_promedio_seg
0,facebook/m2m100_418M,418M,100,25.377944,97.517304,0.975173


modelo m2m100 1.2B

In [16]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model_name_12b = "facebook/m2m100_1.2B"

tokenizer_12b = M2M100Tokenizer.from_pretrained(model_name_12b)
model_12b = M2M100ForConditionalGeneration.from_pretrained(model_name_12b)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_12b.to(device)

print("Modelo cargado:", model_name_12b)
print("Dispositivo:", device)

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modelo cargado: facebook/m2m100_1.2B
Dispositivo: cuda


In [17]:
text = source_texts[6]

tokenizer_12b.src_lang = "en"
encoded_12b = tokenizer_12b(text, return_tensors="pt", truncation=True).to(device)

generated_tokens_12b = model_12b.generate(
    **encoded_12b,
    forced_bos_token_id=tokenizer_12b.get_lang_id("es")
)

translated_text_12b = tokenizer_12b.batch_decode(
    generated_tokens_12b,
    skip_special_tokens=True
)[0]

print("Texto original:")
print(text)

print("\nTraducción generada:")
print(translated_text_12b)

Texto original:
The late owner of this estate was a single man, who lived to a very advanced age, and who for many years of his life, had a constant companion and housekeeper in his sister.

Traducción generada:
El fallecido propietario de esta propiedad era un hombre soltero, que vivió a una edad muy avanzada, y que durante muchos años de su vida, tuvo una constante compañera y ama de casa en su hermana.


In [18]:
predictions_12b = []

inicio_12b = time.time()

for text in source_texts:
    tokenizer_12b.src_lang = "en"
    encoded_12b = tokenizer_12b(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    generated_tokens_12b = model_12b.generate(
        **encoded_12b,
        forced_bos_token_id=tokenizer_12b.get_lang_id("es")
    )

    translated_text_12b = tokenizer_12b.batch_decode(
        generated_tokens_12b,
        skip_special_tokens=True
    )[0]

    predictions_12b.append(translated_text_12b)

fin_12b = time.time()
tiempo_total_12b = fin_12b - inicio_12b

print("Cantidad de traducciones generadas:", len(predictions_12b))
print("Tiempo total de inferencia:", tiempo_total_12b, "segundos")
print("Tiempo promedio por traducción:", tiempo_total_12b / len(predictions_12b), "segundos")

Cantidad de traducciones generadas: 100
Tiempo total de inferencia: 138.96789479255676 segundos
Tiempo promedio por traducción: 1.3896789479255676 segundos


In [19]:
for i in range(5, 10):
    print(f"Ejemplo {i+1}")
    print("EN:", source_texts[i])
    print("REF:", reference_texts[i])
    print("PRED:", predictions_12b[i])
    print("-" * 80)

Ejemplo 6
EN: Their estate was large, and their residence was at Norland Park, in the centre of their property, where, for many generations, they had lived in so respectable a manner as to engage the general good opinion of their surrounding acquaintance.
REF: Su propiedad era de buen tamaño, y en el centro de ella se encontraba la residencia, Norland Park, donde la manera tan digna en que habían vivido por muchas generaciones llegó a granjearles el respeto de todos los conocidos del lugar.
PRED: Su posesión era grande, y su residencia estaba en Norland Park, en el centro de su propiedad, donde, durante muchas generaciones, habían vivido de una manera tan respetable que habían involucrado la buena opinión general de sus conocidos circundantes.
--------------------------------------------------------------------------------
Ejemplo 7
EN: The late owner of this estate was a single man, who lived to a very advanced age, and who for many years of his life, had a constant companion and hous

BLEU

In [20]:
bleu = evaluate.load("sacrebleu")

references_for_bleu = [[ref] for ref in reference_texts]

bleu_result_12b = bleu.compute(
    predictions=predictions_12b,
    references=references_for_bleu
)

print("BLEU:", bleu_result_12b["score"])
print(bleu_result_12b)

BLEU: 28.134686503045867
{'score': 28.134686503045867, 'counts': [1744, 966, 583, 371], 'totals': [2913, 2813, 2713, 2616], 'precisions': [59.8695502917954, 34.34056167792392, 21.489126428308147, 14.181957186544343], 'bp': 1.0, 'sys_len': 2913, 'ref_len': 2905}


In [21]:
summary_df_12b = pd.DataFrame([{
    "modelo": model_name_12b,
    "tamano": "1.2B",
    "num_ejemplos": len(source_texts),
    "bleu": bleu_result_12b["score"],
    "tiempo_total_seg": tiempo_total_12b,
    "tiempo_promedio_seg": tiempo_total_12b / len(source_texts)
}])

summary_df_12b

,modelo,tamano,num_ejemplos,bleu,tiempo_total_seg,tiempo_promedio_seg
0,facebook/m2m100_1.2B,1.2B,100,28.134687,138.967895,1.389679


Liberar espacio

In [22]:
del model
del tokenizer
del model_12b
del tokenizer_12b
torch.cuda.empty_cache()

Modelo nllb-200 600M

In [23]:
model_name_nllb_600m = "facebook/nllb-200-distilled-600M"

tokenizer_nllb_600m = AutoTokenizer.from_pretrained(model_name_nllb_600m)
model_nllb_600m = AutoModelForSeq2SeqLM.from_pretrained(model_name_nllb_600m)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_nllb_600m.to(device)

print("Modelo cargado:", model_name_nllb_600m)
print("Dispositivo:", device)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modelo cargado: facebook/nllb-200-distilled-600M
Dispositivo: cuda


In [24]:
text = source_texts[8]

tokenizer_nllb_600m.src_lang = "eng_Latn"
encoded_nllb_600m = tokenizer_nllb_600m(
    text,
    return_tensors="pt",
    truncation=True
).to(device)

generated_tokens_nllb_600m = model_nllb_600m.generate(
    **encoded_nllb_600m,
    forced_bos_token_id=tokenizer_nllb_600m.convert_tokens_to_ids("spa_Latn")
)

translated_text_nllb_600m = tokenizer_nllb_600m.batch_decode(
    generated_tokens_nllb_600m,
    skip_special_tokens=True
)[0]

print("Texto original:")
print(text)

print("\nTraducción generada:")
print(translated_text_nllb_600m)

Texto original:
In the society of his nephew and niece, and their children, the old Gentleman's days were comfortably spent.

Traducción generada:
En la sociedad de sus sobrinos y sobrinas, y de sus hijos, los días del viejo caballero se pasaron cómodamente.


In [25]:
predictions_nllb_600m = []

inicio_nllb_600m = time.time()

for text in source_texts:
    tokenizer_nllb_600m.src_lang = "eng_Latn"

    encoded_nllb_600m = tokenizer_nllb_600m(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    generated_tokens_nllb_600m = model_nllb_600m.generate(
        **encoded_nllb_600m,
        forced_bos_token_id=tokenizer_nllb_600m.convert_tokens_to_ids("spa_Latn")
    )

    translated_text_nllb_600m = tokenizer_nllb_600m.batch_decode(
        generated_tokens_nllb_600m,
        skip_special_tokens=True
    )[0]

    predictions_nllb_600m.append(translated_text_nllb_600m)

fin_nllb_600m = time.time()
tiempo_total_nllb_600m = fin_nllb_600m - inicio_nllb_600m

print("Cantidad de traducciones generadas:", len(predictions_nllb_600m))
print("Tiempo total de inferencia:", tiempo_total_nllb_600m, "segundos")
print("Tiempo promedio por traducción:", tiempo_total_nllb_600m / len(predictions_nllb_600m), "segundos")

Cantidad de traducciones generadas: 100
Tiempo total de inferencia: 66.19312310218811 segundos
Tiempo promedio por traducción: 0.6619312310218811 segundos


In [26]:
for i in range(4, 11):
    print(f"Ejemplo {i+1}")
    print("EN:", source_texts[i])
    print("REF:", reference_texts[i])
    print("PRED:", predictions_nllb_600m[i])
    print("-" * 80)

Ejemplo 5
EN: The family of Dashwood had long been settled in Sussex.
REF: La familia Dashwood llevaba largo tiempo afincada en Sussex.
PRED: La familia de Dashwood se había establecido en Sussex durante mucho tiempo.
--------------------------------------------------------------------------------
Ejemplo 6
EN: Their estate was large, and their residence was at Norland Park, in the centre of their property, where, for many generations, they had lived in so respectable a manner as to engage the general good opinion of their surrounding acquaintance.
REF: Su propiedad era de buen tamaño, y en el centro de ella se encontraba la residencia, Norland Park, donde la manera tan digna en que habían vivido por muchas generaciones llegó a granjearles el respeto de todos los conocidos del lugar.
PRED: Su finca era grande, y su residencia era en Norland Park, en el centro de su propiedad, donde, durante muchas generaciones, habían vivido de una manera tan respetable que atraía la buena opinión gene

BLEU

In [27]:
bleu_result_nllb_600m = bleu.compute(
    predictions=predictions_nllb_600m,
    references=references_for_bleu
)

print("BLEU:", bleu_result_nllb_600m["score"])
print(bleu_result_nllb_600m)

BLEU: 27.25940559631883
{'score': 27.25940559631883, 'counts': [1725, 933, 561, 351], 'totals': [2889, 2789, 2689, 2591], 'precisions': [59.70924195223261, 33.45285048404446, 20.86277426552622, 13.546893091470475], 'bp': 0.994477059296929, 'sys_len': 2889, 'ref_len': 2905}


In [28]:
summary_df_nllb_600m = pd.DataFrame([{
    "modelo": model_name_nllb_600m,
    "tamano": "600M",
    "num_ejemplos": len(source_texts),
    "bleu": bleu_result_nllb_600m["score"],
    "tiempo_total_seg": tiempo_total_nllb_600m,
    "tiempo_promedio_seg": tiempo_total_nllb_600m / len(source_texts)
}])

summary_df_nllb_600m

,modelo,tamano,num_ejemplos,bleu,tiempo_total_seg,tiempo_promedio_seg
0,facebook/nllb-200-distilled-600M,600M,100,27.259406,66.193123,0.661931


Modelo NLLB 1.3B

In [29]:
del model_nllb_600m
del tokenizer_nllb_600m
torch.cuda.empty_cache()

In [30]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name_nllb_13b = "facebook/nllb-200-1.3B"

tokenizer_nllb_13b = AutoTokenizer.from_pretrained(model_name_nllb_13b)
model_nllb_13b = AutoModelForSeq2SeqLM.from_pretrained(model_name_nllb_13b)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_nllb_13b.to(device)

print("Modelo cargado:", model_name_nllb_13b)
print("Dispositivo:", device)

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modelo cargado: facebook/nllb-200-1.3B
Dispositivo: cuda


In [31]:
text = source_texts[7]

tokenizer_nllb_13b.src_lang = "eng_Latn"
encoded_nllb_13b = tokenizer_nllb_13b(
    text,
    return_tensors="pt",
    truncation=True
).to(device)

generated_tokens_nllb_13b = model_nllb_13b.generate(
    **encoded_nllb_13b,
    forced_bos_token_id=tokenizer_nllb_13b.convert_tokens_to_ids("spa_Latn")
)

translated_text_nllb_13b = tokenizer_nllb_13b.batch_decode(
    generated_tokens_nllb_13b,
    skip_special_tokens=True
)[0]

print("Texto original:")
print(text)

print("\nTraducción generada:")
print(translated_text_nllb_13b)

Texto original:
But her death, which happened ten years before his own, produced a great alteration in his home; for to supply her loss, he invited and received into his house the family of his nephew Mr. Henry Dashwood, the legal inheritor of the Norland estate, and the person to whom he intended to bequeath it.

Traducción generada:
Pero su muerte, que ocurrió diez años antes que la suya, produjo una gran alteración en su hogar; pues para suplir su pérdida, invitó y recibió en su casa a la familia de su sobrino el señor Henry Dashwood, heredero legal de la finca de Norland, y a la persona a quien pretendía legársela.


In [32]:
#traduccion muestra
predictions_nllb_13b = []

inicio_nllb_13b = time.time()

for text in source_texts:
    tokenizer_nllb_13b.src_lang = "eng_Latn"

    encoded_nllb_13b = tokenizer_nllb_13b(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    generated_tokens_nllb_13b = model_nllb_13b.generate(
        **encoded_nllb_13b,
        forced_bos_token_id=tokenizer_nllb_13b.convert_tokens_to_ids("spa_Latn")
    )

    translated_text_nllb_13b = tokenizer_nllb_13b.batch_decode(
        generated_tokens_nllb_13b,
        skip_special_tokens=True
    )[0]

    predictions_nllb_13b.append(translated_text_nllb_13b)

fin_nllb_13b = time.time()
tiempo_total_nllb_13b = fin_nllb_13b - inicio_nllb_13b

print("Cantidad de traducciones generadas:", len(predictions_nllb_13b))
print("Tiempo total de inferencia:", tiempo_total_nllb_13b, "segundos")
print("Tiempo promedio por traducción:", tiempo_total_nllb_13b / len(predictions_nllb_13b), "segundos")

Cantidad de traducciones generadas: 100
Tiempo total de inferencia: 115.30632615089417 segundos
Tiempo promedio por traducción: 1.1530632615089416 segundos


In [33]:
for i in range(6, 12):
    print(f"Ejemplo {i+1}")
    print("EN:", source_texts[i])
    print("REF:", reference_texts[i])
    print("PRED:", predictions_nllb_13b[i])
    print("-" * 80)

Ejemplo 7
EN: The late owner of this estate was a single man, who lived to a very advanced age, and who for many years of his life, had a constant companion and housekeeper in his sister.
REF: El último dueño de esta propiedad había sido un hombre soltero, que alcanzó una muy avanzada edad, y que durante gran parte de su existencia tuvo en su hermana una fiel compañera y ama de casa.
PRED: El difunto propietario de esta finca era un hombre soltero, que vivió hasta una edad muy avanzada, y que durante muchos años de su vida, tuvo una compañera constante y ama de casa en su hermana.
--------------------------------------------------------------------------------
Ejemplo 8
EN: But her death, which happened ten years before his own, produced a great alteration in his home; for to supply her loss, he invited and received into his house the family of his nephew Mr. Henry Dashwood, the legal inheritor of the Norland estate, and the person to whom he intended to bequeath it.
REF: Pero la muert

BLEU

In [34]:
source_texts = [item["translation"]["en"] for item in sample_dataset]
reference_texts = [item["translation"]["es"] for item in sample_dataset]

references_for_bleu = [[ref] for ref in reference_texts]

print("sample_size:", sample_size)
print("source_texts:", len(source_texts))
print("reference_texts:", len(reference_texts))
print("references_for_bleu:", len(references_for_bleu))
print("predictions_nllb_13b:", len(predictions_nllb_13b))

sample_size: 100
source_texts: 100
reference_texts: 100
references_for_bleu: 100
predictions_nllb_13b: 100


In [35]:
bleu_result_nllb_13b = bleu.compute(
    predictions=predictions_nllb_13b,
    references=references_for_bleu
)

print("BLEU:", bleu_result_nllb_13b["score"])
print(bleu_result_nllb_13b)

BLEU: 29.025148877493084
{'score': 29.025148877493084, 'counts': [1787, 996, 615, 397], 'totals': [2949, 2849, 2749, 2651], 'precisions': [60.59681247880638, 34.95963495963496, 22.371771553292106, 14.975480950584686], 'bp': 1.0, 'sys_len': 2949, 'ref_len': 2905}


In [36]:
#resumen
summary_df_nllb_13b = pd.DataFrame([{
    "modelo": model_name_nllb_13b,
    "tamano": "1.3B",
    "num_ejemplos": len(source_texts),
    "bleu": bleu_result_nllb_13b["score"],
    "tiempo_total_seg": tiempo_total_nllb_13b,
    "tiempo_promedio_seg": tiempo_total_nllb_13b / len(source_texts)
}])

summary_df_nllb_13b

,modelo,tamano,num_ejemplos,bleu,tiempo_total_seg,tiempo_promedio_seg
0,facebook/nllb-200-1.3B,1.3B,100,29.025149,115.306326,1.153063


In [37]:
final_summary_df = pd.concat([
    summary_df,
    summary_df_12b,
    summary_df_nllb_600m,
    summary_df_nllb_13b
], ignore_index=True)

final_summary_df

,modelo,tamano,num_ejemplos,bleu,tiempo_total_seg,tiempo_promedio_seg
0,facebook/m2m100_418M,418M,100,25.377944,97.517304,0.975173
1,facebook/m2m100_1.2B,1.2B,100,28.134687,138.967895,1.389679
2,facebook/nllb-200-distilled-600M,600M,100,27.259406,66.193123,0.661931
3,facebook/nllb-200-1.3B,1.3B,100,29.025149,115.306326,1.153063
